# 05 — Fraud-Aware EDA

## Objective

- Analyze human/bot distribution
- Analyze bot presence in graph relationships
- Measure bot contamination in PYMK candidate sets
- Evaluate whether trust filtering is meaningful for this PoC

> Bot accounts are used only as a proxy for suspicious/untrusted accounts. Bot labels must not be interpreted as ground-truth fraud labels.

In [ ]:
# Nhập thư viện cho bot analysis, graph processing và plotting.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import sparse

In [ ]:
# Nạp raw data, xác minh bot label và dùng lại graph semantics của notebook 04.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "MGTAB"
RESULTS_DIR = PROJECT_ROOT / "results" / "eda_05"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def load_tensor(name):
    try:
        return torch.load(DATA_DIR / name, map_location="cpu", weights_only=True)
    except TypeError:
        return torch.load(DATA_DIR / name, map_location="cpu")

edge_index = load_tensor("edge_index.pt").long()
edge_type = load_tensor("edge_type.pt").long()
labels_bot = load_tensor("labels_bot.pt").long().numpy()
num_nodes = len(labels_bot)
src = edge_index[0].numpy()
dst = edge_index[1].numpy()

# Giữ đúng semantics của notebook 04: undirected projection trên toàn bộ relation.
row = np.concatenate([src, dst])
col = np.concatenate([dst, src])
adj = sparse.csr_matrix((np.ones(len(row), dtype=np.int8), (row, col)), shape=(num_nodes, num_nodes))
adj.data[:] = 1
adj.eliminate_zeros()
degree = np.diff(adj.indptr)

unique_labels, label_counts = np.unique(labels_bot, return_counts=True)
assert set(unique_labels.tolist()) <= {0, 1}, f"Unexpected bot labels: {unique_labels}"
# Mapping đã được xác minh theo README chính thức của MGTAB: 0 = human, 1 = bot.
HUMAN_LABEL = 0
BOT_LABEL = 1
human_mask = labels_bot == HUMAN_LABEL
bot_mask = labels_bot == BOT_LABEL

print("Label encoding")
print(f"Human label = {HUMAN_LABEL}; count = {human_mask.sum():,}; percentage = {human_mask.mean():.2%}")
print(f"Bot label = {BOT_LABEL}; count = {bot_mask.sum():,}; percentage = {bot_mask.mean():.2%}")

In [ ]:
# Tạo bảng và biểu đồ phân phối human/bot.
label_distribution = pd.DataFrame({
    "Account Type": ["Human", "Bot"],
    "Count": [int(human_mask.sum()), int(bot_mask.sum())],
})
label_distribution['Percentage'] = label_distribution['Count'] / num_nodes
label_distribution.to_csv(RESULTS_DIR / 'label_distribution.csv', index=False)
display(label_distribution)

ax = label_distribution.plot.bar(x="Account Type", y="Count", legend=False, color=["#4C78A8", "#F58518"])
ax.set_title("MGTAB human/bot distribution")
ax.set_xlabel("Account type")
ax.set_ylabel("Number of accounts")
plt.tight_layout()
plt.show()

## Edge mixing

Vì notebook 04 dùng undirected projection, bảng dưới đây đếm unique undirected pairs. Không diễn giải các nhóm này như quan hệ nhân quả hay hành vi gian lận.

In [ ]:
# Đếm edge mixing human-human, human-bot và bot-bot trên unique pair.
pair_keys = np.minimum(src, dst).astype(np.int64) * num_nodes + np.maximum(src, dst).astype(np.int64)
_, unique_indices = np.unique(pair_keys, return_index=True)
pair_src = src[unique_indices]
pair_dst = dst[unique_indices]
pair_types = np.where(
    (labels_bot[pair_src] == HUMAN_LABEL) & (labels_bot[pair_dst] == HUMAN_LABEL),
    "Human — Human",
    np.where(
        (labels_bot[pair_src] == BOT_LABEL) & (labels_bot[pair_dst] == BOT_LABEL),
        "Bot — Bot",
        "Human — Bot",
    ),
)
mixing = pd.Series(pair_types).value_counts().reindex(["Human — Human", "Human — Bot", "Bot — Bot"], fill_value=0)
edge_mixing = pd.DataFrame({"Edge Type": mixing.index, "Count": mixing.values})
edge_mixing["Percentage"] = edge_mixing["Count"] / edge_mixing["Count"].sum()
edge_mixing.to_csv(RESULTS_DIR / "edge_mixing.csv", index=False)
edge_mixing

In [ ]:
# Tái tạo đúng candidate generation của notebook 04 cho human source users.
def neighbors(node):
    return adj.indices[adj.indptr[node]:adj.indptr[node + 1]]

def generate_candidates(target):
    one_hop = neighbors(target)
    if len(one_hop) == 0:
        return one_hop, np.array([], dtype=np.int64)
    two_hop_pool = np.unique(np.concatenate([neighbors(node) for node in one_hop]))
    excluded = np.concatenate(([target], one_hop))
    candidates = two_hop_pool[~np.isin(two_hop_pool, excluded)]
    return one_hop, candidates

human_users = np.flatnonzero(human_mask)
candidate_rows = []
for i, user_id in enumerate(human_users, start=1):
    one_hop, candidates = generate_candidates(int(user_id))
    bot_count = int(bot_mask[candidates].sum())
    candidate_count = len(candidates)
    candidate_rows.append({
        "user_id": int(user_id),
        "degree": int(len(one_hop)),
        "candidate_count": int(candidate_count),
        "bot_candidate_count": bot_count,
        "human_candidate_count": int(candidate_count - bot_count),
        "bot_candidate_ratio": bot_count / candidate_count if candidate_count else 0.0,
    })
    if i % 500 == 0:
        print(f"processed {i:,}/{len(human_users):,} human users")

candidate_report = pd.DataFrame(candidate_rows)
candidate_report.to_csv(RESULTS_DIR / "candidate_report.csv", index=False)
candidate_report.head()

In [ ]:
# Tổng hợp bot contamination và candidate statistics theo percentile.
def percentiles(series):
    return {
        "mean": float(series.mean()),
        "median": float(series.median()),
        "p90": float(series.quantile(0.90)),
        "p95": float(series.quantile(0.95)),
        "max": float(series.max()),
    }

nonempty = candidate_report["candidate_count"] > 0
with_bot = candidate_report["bot_candidate_count"] > 0
aggregate_rows = [
    {"metric": "human users analyzed", "value": len(candidate_report)},
    {"metric": "human users with no candidate", "value": int((~nonempty).sum())},
    {"metric": "human users with >=1 candidate", "value": int(nonempty.sum())},
    {"metric": "human users with >=1 bot candidate", "value": int(with_bot.sum())},
    {"metric": "percentage with >=1 bot candidate", "value": float(with_bot.mean())},
]
for metric, values in [("bot_candidate_count", candidate_report["bot_candidate_count"]), ("bot_candidate_ratio", candidate_report["bot_candidate_ratio"]), ("candidate_count", candidate_report["candidate_count"])]:
    for statistic, value in percentiles(values).items():
        aggregate_rows.append({"metric": f"{metric}_{statistic}", "value": value})

aggregate_report = pd.DataFrame(aggregate_rows)
aggregate_report.to_csv(RESULTS_DIR / "aggregate_report.csv", index=False)
aggregate_report

In [ ]:
# Vẽ phân phối bot candidate, bot ratio và quan hệ giữa candidate counts.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
candidate_report["bot_candidate_count"].hist(ax=axes[0], bins=40)
candidate_report["bot_candidate_ratio"].hist(ax=axes[1], bins=40)
axes[0].set_title("Bot candidates per human user")
axes[1].set_title("Bot candidate ratio per human user")
for ax in axes:
    ax.set_xlabel("Value")
    ax.set_ylabel("Number of human users")
plt.tight_layout()
plt.show()

candidate_report.plot.scatter(x="candidate_count", y="bot_candidate_count", alpha=0.25, figsize=(6, 4), title="Candidate count vs bot candidates")
plt.show()

In [ ]:
# Chọn vài human users đại diện để kiểm tra candidate contamination trực quan.
eligible = candidate_report[candidate_report["candidate_count"] > 0].copy()
if len(eligible) >= 3:
    example_rows = pd.concat([
        eligible.sort_values("bot_candidate_ratio").head(1),
        eligible.iloc[(eligible["bot_candidate_ratio"] - eligible["bot_candidate_ratio"].median()).abs().argsort()[:1]],
        eligible.sort_values("bot_candidate_ratio").tail(1),
    ]).drop_duplicates("user_id")
    display(example_rows[["user_id", "degree", "candidate_count", "human_candidate_count", "bot_candidate_count", "bot_candidate_ratio"]])
    for user_id in example_rows["user_id"]:
        _, sample_candidates = generate_candidates(int(user_id))
        print(f"user {user_id}: sample candidate IDs={sample_candidates[:10].tolist()}")
else:
    print("Không đủ human users có candidate để tạo 3 ví dụ.")

## Findings

Bảng `label_distribution`, `edge_mixing`, `aggregate_report` và các ví dụ ở trên là cơ sở để trả lời:

1. Bot chiếm bao nhiêu phần trăm labeled users?
2. Human users có thường gặp bot trong candidate set không?
3. Bao nhiêu phần trăm human users có ít nhất một bot candidate?
4. Bot candidate ratio trung bình và median là bao nhiêu?
5. Trust filtering có meaningful trên dataset này không?

Không được kết luận bot là fraud. Phân tích này chỉ đánh giá việc lọc account đáng ngờ dưới dạng proxy trong PoC. MGTAB không phải financial-fraud dataset.